In [ ]:
# =========================
# ULTRA FEATURE ENGINEERING - FOCUS INFLUENCERS VS OBSERVERS
# =========================
import os
import numpy as np
import pandas as pd
from pandas import json_normalize
from tqdm import tqdm
import pickle
from datetime import datetime
import re
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import QuantileTransformer
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("ULTRA FEATURE ENGINEERING - VERSION HARDCORE")
print("="*60)

# =========================
# 1. CHARGEMENT DES DONNÉES
# =========================
print("\n Chargement des fichiers JSONL...")
train_data = pd.read_json('train.jsonl', lines=True)
train_data = json_normalize(train_data.to_dict(orient='records'))
print(f"Train data: {len(train_data)} lignes")

kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))
print(f"Test data: {len(kaggle_data)} lignes")

# =========================
# 2. EXTRACTION TEXTE
# =========================
def extract_full_text(row):
    if 'extended_tweet.full_text' in row.index and pd.notna(row['extended_tweet.full_text']):
        return str(row['extended_tweet.full_text'])
    return str(row.get('text', ''))

train_data['full_text'] = train_data.apply(extract_full_text, axis=1)
kaggle_data['full_text'] = kaggle_data.apply(extract_full_text, axis=1)

# =========================
# 3. FEATURES UTILISATEUR ULTRA-AVANCÉES
# =========================
print("\n👤 Extraction features utilisateur ultra-avancées...")

def extract_ultra_user_features(df):
    features = pd.DataFrame(index=df.index)
    
    def safe_get(col, default=0):
        if col in df.columns:
            return df[col].fillna(default)
        return pd.Series(default, index=df.index)
    
    # Métriques de base
    followers = safe_get('user.followers_count') 
    friends = safe_get('user.friends_count')
    statuses = safe_get('user.statuses_count')
    favourites = safe_get('user.favourites_count')
    listed = safe_get('user.listed_count')
    
    # ==== RATIOS MULTIPLES (CRITICAL) ====
    # Le follower/following ratio est LA feature principale pour identifier les influencers
    features['ff_ratio'] = followers / (friends + 1)
    features['ff_ratio_log'] = np.log1p(features['ff_ratio'])
    features['ff_ratio_sqrt'] = np.sqrt(features['ff_ratio'])
    features['ff_ratio_squared'] = features['ff_ratio'] ** 2
    features['ff_ratio_cubed'] = features['ff_ratio'] ** 3
    
    # Inverse (pour observers)
    features['inverse_ff_ratio'] = friends / (followers + 1)
    features['inverse_ff_ratio_log'] = np.log1p(features['inverse_ff_ratio'])
    
    # Ratios complexes
    features['ff_diff'] = followers - friends
    features['ff_diff_abs'] = np.abs(features['ff_diff'])
    features['ff_diff_log'] = np.log1p(np.abs(features['ff_diff'])) * np.sign(features['ff_diff'])
    
    # Network asymmetry (très discriminant)
    features['network_asymmetry'] = (followers - friends) / (followers + friends + 1)
    features['network_asymmetry_squared'] = features['network_asymmetry'] ** 2
    features['network_asymmetry_abs'] = np.abs(features['network_asymmetry'])
    
    # ==== SCORES D'INFLUENCE MULTIPLES ====
    # Score 1: Combinaison log
    features['influence_score_v1'] = (
        np.log1p(followers) * np.log1p(features['ff_ratio'])
    ) / (np.log1p(friends) + 1)
    
    # Score 2: Puissance
    features['influence_score_v2'] = (
        (followers ** 0.5) * (features['ff_ratio'] ** 0.5)
    )
    
    # Score 3: Exponentiel
    features['influence_score_v3'] = np.log1p(
        followers * features['ff_ratio']
    )
    
    # Score 4: Avec listed
    features['influence_score_v4'] = (
        np.log1p(followers) * 
        np.log1p(listed) * 
        np.log1p(features['ff_ratio'])
    )
    
    # ==== LISTED FEATURES (très important) ====
    features['listed_count'] = listed
    features['listed_log'] = np.log1p(listed)
    features['listed_per_follower'] = listed / (followers + 1)
    features['listed_per_1k_followers'] = (listed * 1000) / (followers + 1)
    features['listed_per_10k_followers'] = (listed * 10000) / (followers + 1)
    features['listed_ratio_log'] = np.log1p(features['listed_per_follower'])
    
    # Listed x ff_ratio (combo puissant)
    features['listed_ff_combo'] = listed * features['ff_ratio']
    features['listed_ff_combo_log'] = np.log1p(features['listed_ff_combo'])
    
    # ==== FEATURES DE TAILLE D'AUDIENCE ====
    features['followers_raw'] = followers
    features['friends_raw'] = friends
    features['followers_log'] = np.log1p(followers)
    features['friends_log'] = np.log1p(friends)
    features['followers_sqrt'] = np.sqrt(followers)
    features['friends_sqrt'] = np.sqrt(friends)
    features['followers_squared'] = followers ** 2
    
    # Catégories d'audience
    features['audience_micro'] = (followers < 1000).astype(int)
    features['audience_small'] = ((followers >= 1000) & (followers < 10000)).astype(int)
    features['audience_medium'] = ((followers >= 10000) & (followers < 100000)).astype(int)
    features['audience_large'] = (followers >= 100000).astype(int)
    
    # ==== FEATURES D'ACTIVITÉ ====
    features['statuses_count'] = statuses
    features['statuses_log'] = np.log1p(statuses)
    features['favourites_count'] = favourites
    features['favourites_log'] = np.log1p(favourites)
    
    # Ratios d'activité
    features['tweets_per_follower'] = statuses / (followers + 1)
    features['tweets_per_friend'] = statuses / (friends + 1)
    features['favs_per_tweet'] = favourites / (statuses + 1)
    features['favs_per_follower'] = favourites / (followers + 1)
    
    # Activity intensity
    features['activity_intensity'] = np.log1p(statuses) / (np.log1p(followers) + 1)
    
    # ==== FEATURES DE PROFIL ====
    features['verified'] = safe_get('user.verified', False).astype(int)
    features['default_profile'] = safe_get('user.default_profile', True).astype(int)
    features['default_profile_image'] = safe_get('user.default_profile_image', True).astype(int)
    features['geo_enabled'] = safe_get('user.geo_enabled', False).astype(int)
    features['has_description'] = safe_get('user.description', '').notna().astype(int)
    features['has_url'] = safe_get('user.url', '').notna().astype(int)
    features['has_location'] = safe_get('user.location', '').notna().astype(int)
    
    # Profile completeness score
    features['profile_complete'] = (
        features['has_description'] +
        features['has_url'] +
        features['has_location'] +
        (1 - features['default_profile_image']) +
        (1 - features['default_profile'])
    )
    
    # Verified x followers (influencers vérifiés)
    features['verified_followers'] = features['verified'] * np.log1p(followers)
    
    # ==== ANCIENNETÉ ====
    if 'user.created_at' in df.columns:
        created_dates = pd.to_datetime(
            df['user.created_at'],
            format='%a %b %d %H:%M:%S %z %Y',
            errors='coerce',
            utc=True
        )
        reference_date = pd.to_datetime('2021-03-17', utc=True)
        account_age_days = (reference_date - created_dates).dt.days.fillna(0)
        
        features['account_age_days'] = account_age_days
        features['account_age_years'] = account_age_days / 365.25
        features['account_age_log'] = np.log1p(account_age_days)
        
        # Growth rates
        features['follower_growth_rate'] = followers / (account_age_days + 1)
        features['friend_growth_rate'] = friends / (account_age_days + 1)
        features['tweet_rate'] = statuses / (account_age_days + 1)
        
        # Growth rate x ff_ratio
        features['growth_influence'] = features['follower_growth_rate'] * features['ff_ratio']
    else:
        for col in ['account_age_days', 'account_age_years', 'account_age_log',
                   'follower_growth_rate', 'friend_growth_rate', 'tweet_rate', 'growth_influence']:
            features[col] = 0
    
    # ==== FEATURES DE QUOTED STATUS ====
    has_quoted = 'quoted_status.user.followers_count' in df.columns
    if has_quoted:
        quoted_followers = safe_get('quoted_status.user.followers_count')
        quoted_friends = safe_get('quoted_status.user.friends_count')
        
        features['quotes_bigger_account'] = (quoted_followers > followers).astype(int)
        features['quotes_influencer'] = ((quoted_followers / (quoted_friends + 1)) > 2).astype(int)
        features['quoted_ff_ratio'] = quoted_followers / (quoted_friends + 1)
    else:
        features['quotes_bigger_account'] = 0
        features['quotes_influencer'] = 0
        features['quoted_ff_ratio'] = 0
    
    return features

user_features_train = extract_ultra_user_features(train_data)
user_features_test = extract_ultra_user_features(kaggle_data)
print(f"✓ {len(user_features_train.columns)} features utilisateur")

# =========================
# 4. FEATURES TWEET ULTRA-AVANCÉES
# =========================
print("\n Extraction features tweet ultra-avancées...")

def extract_ultra_tweet_features(df):
    features = pd.DataFrame(index=df.index)
    
    # Engagement metrics
    retweets = df.get('retweet_count', pd.Series(0, index=df.index)).fillna(0)
    favorites = df.get('favorite_count', pd.Series(0, index=df.index)).fillna(0)
    replies = df.get('reply_count', pd.Series(0, index=df.index)).fillna(0)
    quotes = df.get('quote_count', pd.Series(0, index=df.index)).fillna(0)
    
    # Raw counts
    features['retweet_count'] = retweets
    features['favorite_count'] = favorites
    features['reply_count'] = replies
    features['quote_count'] = quotes
    
    # Log transformations
    features['retweet_log'] = np.log1p(retweets)
    features['favorite_log'] = np.log1p(favorites)
    features['reply_log'] = np.log1p(replies)
    features['quote_log'] = np.log1p(quotes)
    
    # Sqrt transformations
    features['retweet_sqrt'] = np.sqrt(retweets)
    features['favorite_sqrt'] = np.sqrt(favorites)
    
    # Total engagement
    features['total_engagement'] = retweets + favorites + replies + quotes
    features['total_engagement_log'] = np.log1p(features['total_engagement'])
    features['total_engagement_sqrt'] = np.sqrt(features['total_engagement'])
    
    # Engagement ratios
    features['rt_fav_ratio'] = retweets / (favorites + 1)
    features['fav_rt_ratio'] = favorites / (retweets + 1)
    features['reply_total_ratio'] = replies / (features['total_engagement'] + 1)
    features['quote_rt_ratio'] = quotes / (retweets + 1)
    
    # Virality metrics (RT + Quote sont plus viraux)
    features['virality_score'] = (retweets * 2 + quotes * 1.5) / (favorites + replies + 1)
    features['virality_log'] = np.log1p(features['virality_score'])
    
    # Amplification (RT/Fav ratio élevé = amplification)
    features['amplification'] = retweets / (favorites + 1)
    features['amplification_log'] = np.log1p(features['amplification'])
    
    # Conversation vs Broadcast
    features['conversation_metric'] = (replies + quotes) / (features['total_engagement'] + 1)
    features['broadcast_metric'] = (retweets + favorites) / (features['total_engagement'] + 1)
    
    # Type de tweet
    features['is_retweet'] = df.get('retweeted', pd.Series(False, index=df.index)).fillna(False).astype(int)
    features['is_quote'] = df.get('is_quote_status', pd.Series(False, index=df.index)).fillna(False).astype(int)
    features['is_reply'] = df['in_reply_to_status_id'].notna().astype(int) if 'in_reply_to_status_id' in df.columns else 0
    features['is_original'] = ((~features['is_retweet'].astype(bool)) & 
                                (~features['is_reply'].astype(bool))).astype(int)
    
    # Engagement pour original content
    features['original_engagement'] = features['is_original'] * features['total_engagement']
    
    # Sensibilité
    features['possibly_sensitive'] = df.get('possibly_sensitive', pd.Series(False, index=df.index)).fillna(False).astype(int)
    
    # User interaction
    features['favorited'] = df.get('favorited', pd.Series(False, index=df.index)).fillna(False).astype(int)
    features['retweeted'] = df.get('retweeted', pd.Series(False, index=df.index)).fillna(False).astype(int)
    
    return features

tweet_features_train = extract_ultra_tweet_features(train_data)
tweet_features_test = extract_ultra_tweet_features(kaggle_data)
print(f"✓ {len(tweet_features_train.columns)} features tweet")

# =========================
# 5. FEATURES TEXTUELLES HARDCORE
# =========================
print("\n Extraction features textuelles hardcore...")

def extract_hardcore_text_features(df, text_col='full_text'):
    features = pd.DataFrame(index=df.index)
    texts = df[text_col].fillna('')
    
    # ==== LONGUEUR ET STRUCTURE ====
    features['text_length'] = texts.str.len()
    features['text_length_log'] = np.log1p(features['text_length'])
    features['word_count'] = texts.str.split().str.len()
    features['word_count_log'] = np.log1p(features['word_count'])
    features['avg_word_length'] = features['text_length'] / (features['word_count'] + 1)
    
    # Character diversity
    features['unique_chars'] = texts.apply(lambda x: len(set(x)))
    features['char_diversity'] = features['unique_chars'] / (features['text_length'] + 1)
    
    # Word diversity
    features['unique_words'] = texts.apply(lambda x: len(set(x.lower().split())))
    features['word_diversity'] = features['unique_words'] / (features['word_count'] + 1)
    
    # ==== PONCTUATION ====
    features['exclamation'] = texts.str.count('!')
    features['question'] = texts.str.count('\?')
    features['dot'] = texts.str.count('\.')
    features['comma'] = texts.str.count(',')
    features['semicolon'] = texts.str.count(';')
    features['colon'] = texts.str.count(':')
    features['dash'] = texts.str.count('-')
    features['ellipsis'] = texts.str.count('\.\.\.')
    
    # Multiple punctuation (emphase)
    features['multiple_exclamation'] = texts.str.count('!!+')
    features['multiple_question'] = texts.str.count('\?\?+')
    
    # Total punctuation
    features['total_punct'] = (
        features['exclamation'] + features['question'] +
        features['dot'] + features['comma']
    )
    features['punct_density'] = features['total_punct'] / (features['word_count'] + 1)
    
    # Punctuation style (influencers utilisent plus d'exclamations)
    features['exclamation_ratio'] = features['exclamation'] / (features['total_punct'] + 1)
    features['question_ratio'] = features['question'] / (features['total_punct'] + 1)
    
    # ==== STYLE D'ÉCRITURE ====
    features['uppercase_count'] = texts.apply(lambda x: sum(1 for c in x if c.isupper()))
    features['uppercase_ratio'] = features['uppercase_count'] / (features['text_length'] + 1)
    features['digit_count'] = texts.apply(lambda x: sum(1 for c in x if c.isdigit()))
    features['digit_ratio'] = features['digit_count'] / (features['text_length'] + 1)
    features['space_count'] = texts.str.count(' ')
    features['space_ratio'] = features['space_count'] / (features['text_length'] + 1)
    
    # ALL CAPS words (shouting)
    features['allcaps_words'] = texts.apply(
        lambda x: sum(1 for word in x.split() if word.isupper() and len(word) > 1)
    )
    features['allcaps_ratio'] = features['allcaps_words'] / (features['word_count'] + 1)
    
    # ==== EMOJIS ====
    features['emoji_count'] = texts.apply(lambda x: sum(1 for c in x if ord(c) > 127000))
    features['has_emoji'] = (features['emoji_count'] > 0).astype(int)
    features['emoji_density'] = features['emoji_count'] / (features['word_count'] + 1)
    
    # ==== ENTITÉS ====
    for prefix in ['entities', 'extended_tweet.entities']:
        if f'{prefix}.hashtags' in df.columns:
            features['hashtag_count'] = df[f'{prefix}.hashtags'].apply(
                lambda x: len(x) if isinstance(x, list) else 0
            )
        if f'{prefix}.user_mentions' in df.columns:
            features['mention_count'] = df[f'{prefix}.user_mentions'].apply(
                lambda x: len(x) if isinstance(x, list) else 0
            )
        if f'{prefix}.urls' in df.columns:
            features['url_count'] = df[f'{prefix}.urls'].apply(
                lambda x: len(x) if isinstance(x, list) else 0
            )
        if f'{prefix}.media' in df.columns:
            features['media_count'] = df[f'{prefix}.media'].apply(
                lambda x: len(x) if isinstance(x, list) else 0
            )
    
    for col in ['hashtag_count', 'mention_count', 'url_count', 'media_count']:
        if col not in features.columns:
            features[col] = 0
        else:
            features[col] = features[col].fillna(0)
    
    # Entités transformations
    features['hashtag_log'] = np.log1p(features['hashtag_count'])
    features['mention_log'] = np.log1p(features['mention_count'])
    features['url_log'] = np.log1p(features['url_count'])
    features['media_log'] = np.log1p(features['media_count'])
    
    # Présence binaire
    features['has_hashtag'] = (features['hashtag_count'] > 0).astype(int)
    features['has_mention'] = (features['mention_count'] > 0).astype(int)
    features['has_url'] = (features['url_count'] > 0).astype(int)
    features['has_media'] = (features['media_count'] > 0).astype(int)
    
    # Ratios
    features['hashtag_ratio'] = features['hashtag_count'] / (features['word_count'] + 1)
    features['mention_ratio'] = features['mention_count'] / (features['word_count'] + 1)
    features['url_ratio'] = features['url_count'] / (features['word_count'] + 1)
    features['media_ratio'] = features['media_count'] / (features['word_count'] + 1)
    
    # Content richness
    features['entity_count'] = (
        features['hashtag_count'] + 
        features['mention_count'] + 
        features['url_count'] + 
        features['media_count']
    )
    features['entity_density'] = features['entity_count'] / (features['word_count'] + 1)
    
    # Promotional content (URLs + hashtags)
    features['promotional_score'] = (
        features['url_count'] * 2 + 
        features['hashtag_count']
    ) / (features['word_count'] + 1)
    
    # Social engagement (mentions)
    features['social_score'] = features['mention_count'] / (features['word_count'] + 1)
    
    # ==== LANGUE ====
    lang = df.get('lang', pd.Series('', index=df.index))
    features['lang_fr'] = (lang == 'fr').astype(int)
    features['lang_en'] = (lang == 'en').astype(int)
    features['lang_und'] = (lang == 'und').astype(int)
    features['lang_other'] = ((lang != 'fr') & (lang != 'en') & (lang != 'und')).astype(int)
    
    # ==== STYLE CONVERSATIONNEL VS BROADCAST ====
    # Observers : plus de mentions, questions, replies
    reply_count_value = df.get('reply_count', pd.Series(0, index=df.index)).fillna(0)
    features['conversation_signals'] = (
        features['mention_count'] + 
        features['question'] +
        reply_count_value
    )
    
    # Influencers : plus de hashtags, URLs, media, exclamations
    features['broadcast_signals'] = (
        features['hashtag_count'] * 1.5 + 
        features['url_count'] * 2 +
        features['media_count'] * 2 +
        features['exclamation']
    )
    
    features['conversation_score'] = features['conversation_signals'] / (features['word_count'] + 1)
    features['broadcast_score'] = features['broadcast_signals'] / (features['word_count'] + 1)
    features['broadcast_conv_ratio'] = features['broadcast_score'] / (features['conversation_score'] + 0.1)
    
    return features

text_features_train = extract_hardcore_text_features(train_data)
text_features_test = extract_hardcore_text_features(kaggle_data)
print(f"✓ {len(text_features_train.columns)} features textuelles")

# =========================
# 6. FEATURES COMBINÉES ULTRA-AVANCÉES
# =========================
print("\n🔗 Création features combinées ultra-avancées...")

def create_ultra_combined_features(user_feats, tweet_feats, text_feats):
    combined = pd.DataFrame(index=user_feats.index)
    
    # ==== ENGAGEMENT NORMALISÉ PAR AUDIENCE ====
    # Très important : normaliser l'engagement par la taille de l'audience
    followers = user_feats['followers_raw'] + 1
    friends = user_feats['friends_raw'] + 1
    
    # Engagement rate (critique)
    combined['eng_rate'] = tweet_feats['total_engagement'] / followers
    combined['eng_rate_log'] = np.log1p(combined['eng_rate'])
    combined['eng_rate_sqrt'] = np.sqrt(combined['eng_rate'])
    
    # Retweet rate (influencers ont un meilleur RT rate)
    combined['rt_rate'] = tweet_feats['retweet_count'] / followers
    combined['rt_rate_log'] = np.log1p(combined['rt_rate'])
    
    # Favorite rate
    combined['fav_rate'] = tweet_feats['favorite_count'] / followers
    combined['fav_rate_log'] = np.log1p(combined['fav_rate'])
    
    # Engagement per log follower
    combined['eng_per_log_follower'] = (
        tweet_feats['total_engagement'] / (np.log1p(followers))
    )
    
    # ==== REACH ET AMPLIFICATION ====
    combined['potential_reach'] = followers * (1 + tweet_feats['retweet_count'])
    combined['potential_reach_log'] = np.log1p(combined['potential_reach'])
    
    combined['amplification_factor'] = tweet_feats['retweet_count'] / followers
    combined['amplification_log'] = np.log1p(combined['amplification_factor'])
    
    # Viral coefficient
    combined['viral_coef'] = (
        tweet_feats['retweet_count'] + tweet_feats['quote_count']
    ) / followers
    
    # ==== INFLUENCE X ENGAGEMENT ====
    combined['influence_engagement'] = (
        user_feats['ff_ratio'] * 
        np.log1p(tweet_feats['total_engagement'])
    )
    
    combined['influence_rt'] = (
        user_feats['ff_ratio'] * 
        np.log1p(tweet_feats['retweet_count'])
    )
    
    combined['listed_engagement'] = (
        user_feats['listed_per_follower'] * 
        tweet_feats['total_engagement']
    )
    
    # ==== AUTHORITY SCORES ====
    combined['authority_v1'] = (
        user_feats['ff_ratio'] * 
        user_feats['listed_per_follower'] * 
        np.log1p(followers)
    )
    
    combined['authority_v2'] = (
        user_feats['influence_score_v1'] * 
        combined['eng_rate']
    )
    
    combined['authority_v3'] = (
        user_feats['network_asymmetry'] * 
        np.log1p(user_feats['listed_count']) *
        combined['rt_rate']
    )
    
    # ==== CONTENT QUALITY INDICATORS ====
    # Influencers : contenu plus court, plus d'entités
    combined['content_efficiency'] = (
        text_feats['entity_count'] / (text_feats['word_count'] + 1)
    ) * np.log1p(tweet_feats['total_engagement'])
    
    combined['content_reach'] = (
        text_feats['broadcast_score'] * 
        combined['potential_reach_log']
    )
    
    combined['content_length_per_follower'] = (
        text_feats['word_count'] / np.log1p(followers)
    )
    
    # ==== STYLE X ENGAGEMENT ====
    combined['broadcast_engagement'] = (
        text_feats['broadcast_score'] * 
        tweet_feats['total_engagement_log']
    )
    
    combined['conversation_engagement'] = (
        text_feats['conversation_score'] * 
        tweet_feats['reply_count']
    )
    
    # ==== RATIOS CRITIQUES ====
    # Engagement efficiency (engagement par follower ET par ami)
    combined['follower_efficiency'] = (
        tweet_feats['total_engagement'] / followers
    )
    
    combined['friend_efficiency'] = (
        tweet_feats['total_engagement'] / friends
    )
    
    combined['efficiency_ratio'] = (
        combined['follower_efficiency'] / (combined['friend_efficiency'] + 0.001)
    )
    
    # Network value
    combined['network_value'] = (
        user_feats['ff_ratio'] * 
        user_feats['listed_count'] * 
        combined['eng_rate']
    )
    
    # ==== INTERACTIONS COMPLEXES ====
    # FF ratio x engagement x content
    combined['mega_score_v1'] = (
        user_feats['ff_ratio_log'] * 
        combined['eng_rate_log'] *
        text_feats['broadcast_score']
    )
    
    combined['mega_score_v2'] = (
        user_feats['influence_score_v1'] * 
        combined['rt_rate_log'] *
        (1 - text_feats['conversation_score'])
    )
    
    combined['mega_score_v3'] = (
        user_feats['network_asymmetry'] * 
        user_feats['listed_log'] *
        combined['viral_coef']
    )
    
    # Original content performance (influencers postent plus d'original)
    combined['original_performance'] = (
        tweet_feats['is_original'] * 
        tweet_feats['total_engagement'] / followers
    )
    
    return combined

combined_features_train = create_ultra_combined_features(
    user_features_train, tweet_features_train, text_features_train
)
combined_features_test = create_ultra_combined_features(
    user_features_test, tweet_features_test, text_features_test
)
print(f"✓ {len(combined_features_train.columns)} features combinées")

# =========================
# 7. ASSEMBLAGE
# =========================
print("\n🔨 Assemblage du dataset final...")

X_train = pd.concat([
    user_features_train,
    tweet_features_train,
    text_features_train,
    combined_features_train
], axis=1)

X_test = pd.concat([
    user_features_test,
    tweet_features_test,
    text_features_test,
    combined_features_test
], axis=1)

# Gestion valeurs infinies et NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

y_train = train_data['label'].values

print(f"Features totales: {X_train.shape[1]}")
print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

# =========================
# 8. FEATURE SELECTION AGRESSIVE
# =========================
print("\n Feature selection basée sur les corrélations...")

y_train_series = pd.Series(y_train, index=X_train.index)
correlations = X_train.corrwith(y_train_series).abs().sort_values(ascending=False)

print("\n Top 50 features par corrélation absolue:")
print(correlations.head(50))

# Supprimer les features avec corrélation nulle
zero_corr_features = correlations[correlations == 0].index.tolist()
if len(zero_corr_features) > 0:
    print(f"\n  Suppression de {len(zero_corr_features)} features avec corrélation nulle")
    X_train = X_train.drop(columns=zero_corr_features)
    X_test = X_test.drop(columns=zero_corr_features)

# =========================
# 9. TRANSFORMATIONS ADDITIONNELLES
# =========================
print("\n Application de transformations quantile...")

# Appliquer QuantileTransformer sur les top features pour les rendre plus normales
top_features = correlations.head(30).index.tolist()
top_features = [f for f in top_features if f in X_train.columns]

qt = QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
X_train_qt = X_train.copy()
X_test_qt = X_test.copy()

X_train_qt[top_features] = qt.fit_transform(X_train[top_features])
X_test_qt[top_features] = qt.transform(X_test[top_features])

print(f"Transformation quantile appliquée sur {len(top_features)} features")

# =========================
# 10. SAUVEGARDE
# =========================
print("\n Sauvegarde des datasets...")

# Version normale
X_train.to_csv('X_train_ultra.csv', index=False)
X_test.to_csv('X_test_ultra.csv', index=False)

# Version quantile transformée
X_train_qt.to_csv('X_train_ultra_qt.csv', index=False)
X_test_qt.to_csv('X_test_ultra_qt.csv', index=False)

pd.DataFrame({'label': y_train}).to_csv('y_train.csv', index=False)
pd.DataFrame({'challenge_id': kaggle_data['challenge_id']}).to_csv('test_ids.csv', index=False)

train_data[['full_text']].to_csv('train_texts.csv', index=False)
kaggle_data[['challenge_id', 'full_text']].to_csv('test_texts.csv', index=False)

with open('feature_names_ultra.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)

correlations.to_csv('feature_correlations_ultra.csv', index=False)

print("\n ULTRA FEATURE ENGINEERING TERMINÉ")
print(f"Features finales: {X_train.shape[1]}")
print("="*60)